# Обучение моделей

## Импорт библиотек

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Добавляем путь к src
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.data_loader import DataLoader
from src.preprocess import Preprocessor
from src.features import FeatureEngineer

## Загрузка данных

In [2]:
# Загружаем датасет
PROJECT_ROOT = project_root
DATASET_PATH = PROJECT_ROOT / "datasets" / "Credit Risk Data.csv"
KAGGLE_DS = "alexdister/credit-risk-dataset"

df = DataLoader.load(DATASET_PATH, KAGGLE_DS)

In [3]:
df.head()

,client_ID,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,...,city_latitude,city_longitude,employment_type,loan_term_months,loan_to_income_ratio,other_debt,debt_to_income_ratio,open_accounts,credit_utilization_ratio,past_delinquencies
0,CUST_00001,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,...,43.6532,-79.3832,Self-employed,36,0.593220,8402.453850,0.735635,14,0.495557,0
1,CUST_00002,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,...,43.6532,-79.3832,Full-time,36,0.104167,1607.802794,0.271646,10,0.585436,3
2,CUST_00003,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,...,51.6214,-3.9436,Full-time,36,0.572917,2760.505633,0.860469,14,0.750732,0
3,CUST_00004,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,...,49.2827,-123.1207,Part-time,12,0.534351,7155.286150,0.643592,15,0.379333,0
4,CUST_00005,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,...,42.8864,-78.8784,Part-time,36,0.643382,15626.153440,0.930628,4,0.228103,0


## Разделение на выборки

In [4]:
# Разделение на признаки и целевую переменную
X = df.drop(columns=["loan_status"])
y = df["loan_status"]

print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (32581, 28), y shape: (32581,)


In [5]:
# Разделение на обучающую, валидационную и тестовую выборки 

# 1) Train + Val vs Test (80% / 20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 2) Train vs Val (80% / 20% от train_val → 64% / 16% от всего)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.2,
    random_state=42,
    stratify=y_train_val
)

print(f"Train: {X_train.shape[0]} записей")
print(f"Val:   {X_val.shape[0]} записей")
print(f"Test:  {X_test.shape[0]} записей")
print(f"Доля дефолтов в train: {y_train.mean():.4f}")
print(f"Доля дефолтов в val:   {y_val.mean():.4f}")
print(f"Доля дефолтов в test:  {y_test.mean():.4f}")

Train: 20851 записей
Val:   5213 записей
Test:  6517 записей
Доля дефолтов в train: 0.2182
Доля дефолтов в val:   0.2181
Доля дефолтов в test:  0.2182
